In [ ]:
"""
07a - Export the 30 m feature stack from Earth Engine to Drive

Exports one GeoTIFF per available month covering the full region of interest at
30 m, each carrying 46 feature bands in the order defined by config.MODEL_FEATURES:
5 raw bands, 7 spectral indices, 7 z-score anomalies, 7 raw anomalies, and 20
land-cover one-hot bands (10 for the previous year, 10 for the current year). The
land-cover bands are constant within a calendar year, matching the way notebook 02
sampled them for training.

The remaining 6 of the 52 model features (4 delta features and 2 harmonic features)
are derived locally in notebook 07b, since they need no server-side computation.

Total output: roughly 35 GB across 106 files.
Runtime: 2 to 3 minutes to submit; 4 to 6 hours of Earth Engine backend execution.
"""

'\nNotebook 07a v4 — Export the feature stack from GEE to Drive at 30 m\n\nPurpose:\n    Export monthly feature rasters covering the full ROI at 30 m. Each\n    available month is exported as a separate GeoTIFF, with all 46\n    feature bands in the order defined by config.MODEL_FEATURES.\n\nWhat changes from v2:\n    * Uses the v3 S2 collection (starts 2016) and v3 baselines (2016-2017)\n    * Each monthly raster now contains 46 bands: 26 spectral + 20 LC one-hot\n    * The LC bands are the same for every month of a given calendar year\n      (LC_prev = LC of year-1, LC_curr = LC of year), matching how\n      Notebook 02 sampled them for training\n\nTotal output: ~35 GB across 106 files (46 bands × 30 m × ROI area)\nRuntime:\n    - Task submission: 2-3 minutes\n    - GEE backend execution: 4-6 hours (runs server-side)\n'

# 07a - Export the 46-band feature stack

**Purpose** - export the monthly feature rasters for the whole region of interest at
30 m, so that inference can be run block-wise over the full ROI rather than over the
sampled pixels alone.

**Main inputs** - the Earth Engine assets built in notebook 01 (Sentinel-2 collection,
seasonal baselines, land-cover layer).

**Main outputs** - one 46-band GeoTIFF per available month, written to Drive; the task
manifest `data/raw/export_tasks_v4_30m.json`.

**Role in the thesis workflow** - the first step of the supplementary 30 m full-ROI
inference chain (07a to 07b to 07c). The final regional analysis in the thesis runs at
10 m in `pipeline_10m/`; this 30 m chain produced the rasters behind the appendix
diagnostics and the Section 4.12 tables.

In [1]:
# --- Environment auto-detection ---
import sys, os
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    THESIS_ROOT = "/content/drive/MyDrive/thesisv4"
else:
    THESIS_ROOT = os.path.expanduser("~/thesisv4")

sys.path.insert(0, THESIS_ROOT)
print(f"=== Environment: {'COLAB' if IS_COLAB else 'LOCAL'} ===")
print(f"Workspace: {THESIS_ROOT}")

Mounted at /content/drive
=== Environment: COLAB ===
Workspace: /content/drive/MyDrive/thesisv4


In [2]:
import sys, importlib
# sys.path already set by auto-detect cell above
import config
importlib.reload(config)
config.ensure_dirs()
config.print_summary()

CONFIG SUMMARY  (run_v4)
Environment:        Colab
Workspace root:     /content/drive/MyDrive/thesisv4
GEE project:        bar-project-457815
Scale:              10 m
CRS:                EPSG:32636
Analysis period:    2017-01 to 2025-12
Analysis months:    108
Baseline years:     [2016, 2017]
Total features:     52
Sampling targets:
  n_no_fires              304,461
  n_low_recurrence        300,032
  n_medium_recurrence       4,050
  n_high_recurrence           379
  quality_threshold           0.2
  TOTAL                   608,922
Model:              d_model=128, n_head=4, d_k=32
Training:           batch_size=64, epochs=50


In [3]:
import ee
import json
from pathlib import Path

try:
    ee.Initialize(project=config.GEE["project"])
except Exception:
    ee.Authenticate()
    ee.Initialize(project=config.GEE["project"])
print("GEE initialised.")

GEE initialised.


## 1. Configuration

In [4]:
EXPORT_SCALE = 30
EXPORT_CRS   = "EPSG:32636"

# Output folder name on Drive (GEE places it at MyDrive/<this folder>/)
DRIVE_FOLDER = "feature_stack_v4_30m"

# Where the exported rasters end up for THIS environment. On Colab that is the
# mounted Drive; locally GEE still writes to Drive, so point at the directory
# you sync them down into (notebook 07b reads from the same place).
if IS_COLAB:
    FEATURE_STACK_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    FEATURE_STACK_DIR = Path(THESIS_ROOT) / "data" / "raw" / DRIVE_FOLDER
FEATURE_STACK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Local feature-stack dir: {FEATURE_STACK_DIR}")

roi           = ee.FeatureCollection(config.GEE["roi"])
roi_geom      = roi.geometry()
s2_collection = ee.ImageCollection(config.GEE["ic_s2"])
baseline_mean = ee.ImageCollection(config.GEE["ic_baseline_mean"])
baseline_std  = ee.ImageCollection(config.GEE["ic_baseline_std"])
lc_collection = ee.ImageCollection(config.GEE["landcover_yearly"])

print(f"Export resolution:  {EXPORT_SCALE} m")
print(f"Output folder:      MyDrive/{DRIVE_FOLDER}")
print(f"S2 collection size: {s2_collection.size().getInfo()}")
print(f"LC collection size: {lc_collection.size().getInfo()}")

Local feature-stack dir: /content/drive/MyDrive/feature_stack_v4_30m
Export resolution:  30 m
Output folder:      MyDrive/feature_stack_v4_30m
S2 collection size: 117
LC collection size: 8


## 2. Identify which months exist in the S2 collection

In [7]:
all_months = config.get_month_list()
T = len(all_months)
print(f"Canonical timeline: T = {T} months (2017-2025)")

Canonical timeline: T = 108 months (2017-2025)


In [9]:
def list_existing_months():
    """Return the set of (year, month) tuples present in the S2 collection."""
    s2_info = s2_collection.aggregate_array("system:index").getInfo()
    existing = set()
    for idx in s2_info:
        parts = idx.split("_")
        try:
            y, m = int(parts[-2]), int(parts[-1])
            existing.add((y, m))
        except (ValueError, IndexError):
            continue
    return existing


existing_ym = list_existing_months()
print(f"Months found in S2 collection: {len(existing_ym)}")

missing = [ym for ym in all_months if ym not in existing_ym]
print(f"Missing months ({len(missing)}): {missing}")

months_to_export = [
    (m_idx, ym) for m_idx, ym in enumerate(all_months)
    if ym in existing_ym
]
print(f"Months to export: {len(months_to_export)}")

Months found in S2 collection: 117
Missing months (0): []
Months to export: 108


## 3. Build the per-month feature image

46 bands are exported from Earth Engine. The 4 delta features and the 2 harmonic
features are computed in notebook 07b in Python instead, because they need no
server-side computation and are cheap to derive locally.

Bands exported here:
* 5 raw bands (B4, B8, B8A, B11, B12)
* 7 spectral indices (NDVI, EVI, NDMI, NBR, MIRBI, BSI, BAIS2)
* 7 z-score anomalies (`<index>_zscore`)
* 7 raw anomalies (`<index>_anomaly`)
* 10 previous-year land-cover one-hot bands
* 10 current-year land-cover one-hot bands

In [ ]:
RAW_BANDS     = config.RAW_BANDS
INDEX_BANDS   = config.INDICES
ANOMALY_BASES = config.ANOMALY_SOURCE_INDICES

# The subset of MODEL_FEATURES that we export from GEE (46 bands).
# delta_* and month_* features are computed in Notebook 07b.
GEE_FEATURES = [
    f for f in config.MODEL_FEATURES
    if f not in config.DELTA_FEATURES and f not in config.HARMONIC_FEATURES
]

print(f"config.MODEL_FEATURES length: {len(config.MODEL_FEATURES)}")
print(f"  Full model features:  {len(config.MODEL_FEATURES)} "
      f"(includes delta + harmonic computed in 07b)")
print(f"  GEE-exported bands:   {len(GEE_FEATURES)}")
print(f"  First 5: {GEE_FEATURES[:5]}")
print(f"  Last 5:  {GEE_FEATURES[-5:]}")

config.MODEL_FEATURES length: 52
  Full model features:  52 (includes delta + harmonic computed in 07b)
  GEE-exported bands:   46
  First 5: ['B4', 'B8', 'B8A', 'B11', 'B12']
  Last 5:  ['lc_curr_covered_agriculture', 'lc_curr_built_up', 'lc_curr_bare', 'lc_curr_fallow', 'lc_curr_water']


In [ ]:
def lc_year_index(year):
    """Clamp the LC lookup year to the range available in the dataset."""
    return max(int(year), 2018)


def lc_image_for_year(year):
    """Return the LC image whose system:index matches the target year."""
    idx = str(lc_year_index(year))
    return ee.Image(lc_collection.filter(
        ee.Filter.eq("system:index", idx)).first()).select("b1")


def lc_onehot_stack(year, feature_names):
    """Stack the one-hot LC bands for the given calendar year."""
    lc_img = lc_image_for_year(year)
    bands = []
    for code, feat_name in zip(config.LC_CODES, feature_names):
        bands.append(lc_img.eq(code).toFloat().rename(feat_name))
    return ee.Image.cat(bands)

In [ ]:
def add_indices(img_ref):
    """Attach the seven spectral indices to a raw Sentinel-2 image.

    The S2 assets contain only 5 raw bands + pct_valid to save GEE storage.
    Indices are computed on-the-fly here, matching Notebook 02's logic.
    """
    eps = ee.Image.constant(1e-6)

    b4  = img_ref.select("B4")
    b8  = img_ref.select("B8")
    b8a = img_ref.select("B8A")
    b11 = img_ref.select("B11")
    b12 = img_ref.select("B12")

    ndvi = img_ref.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndmi = img_ref.normalizedDifference(["B8", "B11"]).rename("NDMI")
    nbr  = img_ref.normalizedDifference(["B8", "B12"]).rename("NBR")

    evi = (b8.subtract(b4).multiply(2.5)
             .divide(b8.add(b4.multiply(2.4)).add(1).add(eps))
             .rename("EVI"))

    mirbi = (b12.multiply(10).subtract(b11.multiply(9.8)).add(2)).rename("MIRBI")

    bsi = ((b11.add(b4).subtract(b8.add(b8a)))
           .divide(b11.add(b4).add(b8).add(b8a).add(eps))
           .rename("BSI"))

    term1 = ee.Image(1).subtract(
        b11.multiply(b12).multiply(b8a).divide(b4.add(eps)).sqrt())
    term2 = b12.subtract(b8a).divide(b12.add(b8a).add(eps).sqrt()).add(1)
    bais2 = term1.multiply(term2).rename("BAIS2")

    return (img_ref.select(config.RAW_BANDS)
                    .addBands([ndvi, evi, ndmi, nbr, mirbi, bsi, bais2])
                    .toFloat())


def build_monthly_feature_image(year, month):
    """Build one Image with 46 bands in GEE_FEATURES order.

    Delta and harmonic features are added later, in Notebook 07b, when
    the raster is loaded into a numpy tensor.
    """
    img_m = (s2_collection
             .filter(ee.Filter.eq("year", year))
             .filter(ee.Filter.eq("month", month))
             .first())
    img_m = ee.Image(img_m)

    # Compute indices on-the-fly (S2 asset only has raw + pct_valid)
    img_with_indices = add_indices(img_m)

    base_mean_m = ee.Image(baseline_mean.filter(
        ee.Filter.eq("month", month)).first())
    base_std_m  = ee.Image(baseline_std.filter(
        ee.Filter.eq("month", month)).first())

    # Raw + index bands
    raw_and_index = img_with_indices.select(RAW_BANDS + INDEX_BANDS)

    # Anomaly bands: z-score and raw anomaly
    # (baseline assets already contain both raw bands and indices per Notebook 01)
    anomaly_imgs = []
    for b in ANOMALY_BASES:
        z = (img_with_indices.select(b).subtract(base_mean_m.select(b))
                 .divide(base_std_m.select(b).max(1e-6))
                 .rename(f"{b}_zscore"))
        a = (img_with_indices.select(b).subtract(base_mean_m.select(b))
                 .rename(f"{b}_anomaly"))
        anomaly_imgs.append(z)
        anomaly_imgs.append(a)

    # Land-cover one-hot bands
    lc_prev = lc_onehot_stack(year - 1, config.LC_FEATURES_PREV)
    lc_curr = lc_onehot_stack(year,     config.LC_FEATURES_CURR)

    feature_img = raw_and_index
    for a in anomaly_imgs:
        feature_img = feature_img.addBands(a)
    feature_img = feature_img.addBands(lc_prev).addBands(lc_curr)

    # Reorder to exactly match the GEE-exported subset of MODEL_FEATURES
    feature_img = feature_img.select(GEE_FEATURES)

    return feature_img.clip(roi_geom).toFloat()

In [ ]:
# Verify on a single month before launching all exports
m_idx_test, (y_test, m_test) = months_to_export[5]
test_img   = build_monthly_feature_image(y_test, m_test)
test_bands = test_img.bandNames().getInfo()

print(f"Test image: year={y_test}, month={m_test} (m_idx={m_idx_test})")
print(f"  Band count: {len(test_bands)}")
print(f"  First 5: {test_bands[:5]}")
print(f"  Last 5:  {test_bands[-5:]}")

assert test_bands == list(GEE_FEATURES), \
    "Band order mismatch — check GEE_FEATURES against config.MODEL_FEATURES"
print("  Band order matches GEE_FEATURES (46 bands)")

Test image: year=2017, month=6 (m_idx=5)
  Band count: 46
  First 5: ['B4', 'B8', 'B8A', 'B11', 'B12']
  Last 5:  ['lc_curr_covered_agriculture', 'lc_curr_built_up', 'lc_curr_bare', 'lc_curr_fallow', 'lc_curr_water']
  Band order matches GEE_FEATURES (46 bands)


## 4. Submit export tasks

One `Export.image.toDrive` task per available month, queued and run server-side. The
filename encodes the canonical month index so notebook 07b can place each file at its
correct position in the T = 108 sequence.

In [ ]:
submitted_tasks = []

print(f"Submitting {len(months_to_export)} export tasks...")

for m_idx, (year, month) in months_to_export:
    feature_img = build_monthly_feature_image(year, month)

    task = ee.batch.Export.image.toDrive(
        image          = feature_img,
        description    = f"feature_v4_m{m_idx:03d}_{year}_{month:02d}",
        folder         = DRIVE_FOLDER,
        fileNamePrefix = f"features_m{m_idx:03d}",
        region         = roi_geom,
        scale          = EXPORT_SCALE,
        crs            = EXPORT_CRS,
        maxPixels      = 1e10,
        fileFormat     = "GeoTIFF",
        formatOptions  = {"cloudOptimized": True},
    )
    task.start()

    submitted_tasks.append({
        "m_idx":    m_idx,
        "year":     year,
        "month":    month,
        "task_id":  task.id,
        "filename": f"features_m{m_idx:03d}.tif",
    })

    if m_idx % 10 == 0:
        print(f"  Submitted task for m_idx={m_idx} ({year}-{month:02d})")

print(f"\nAll {len(submitted_tasks)} tasks submitted.")
print(f"Skipped months (will be NaN-padded in inference): {missing}")
print(f"Monitor at: https://code.earthengine.google.com/tasks")

# Save the registry
registry_path = os.path.join(
    THESIS_ROOT, "data", "raw", "export_tasks_v4_30m.json")
with open(registry_path, "w") as f:
    json.dump({
        "submitted_tasks": submitted_tasks,
        "missing_months":  missing,
        "scale":           EXPORT_SCALE,
        "folder":          DRIVE_FOLDER,
        "n_bands":         len(GEE_FEATURES),
        "n_model_features": len(config.MODEL_FEATURES),
        "note":            "delta_* and month_* features are added in Notebook 07b",
    }, f, indent=2)
print(f"Task registry saved to: {registry_path}")

Submitting 108 export tasks...
  Submitted task for m_idx=0 (2017-01)
  Submitted task for m_idx=10 (2017-11)
  Submitted task for m_idx=20 (2018-09)
  Submitted task for m_idx=30 (2019-07)
  Submitted task for m_idx=40 (2020-05)
  Submitted task for m_idx=50 (2021-03)
  Submitted task for m_idx=60 (2022-01)
  Submitted task for m_idx=70 (2022-11)
  Submitted task for m_idx=80 (2023-09)
  Submitted task for m_idx=90 (2024-07)
  Submitted task for m_idx=100 (2025-05)

All 108 tasks submitted.
Skipped months (will be NaN-padded in inference): []
Monitor at: https://code.earthengine.google.com/tasks
Task registry saved to: /home/od/thesisv4/data/raw/export_tasks_v4_30m.json


## 5. Monitor progress

In [ ]:
def summarise_status():
    summary  = {"COMPLETED": 0, "RUNNING": 0, "READY": 0,
                "FAILED": 0, "CANCELLED": 0, "UNKNOWN": 0}
    failures = []
    for task_info in submitted_tasks:
        try:
            state = ee.data.getTaskStatus(task_info["task_id"])[0]["state"]
            summary[state] = summary.get(state, 0) + 1
            if state == "FAILED":
                failures.append((task_info["m_idx"],
                                 task_info["year"], task_info["month"]))
        except Exception:
            summary["UNKNOWN"] += 1
    return summary, failures

In [ ]:
summary, failures = summarise_status()
print(f"Status across {len(submitted_tasks)} tasks:")
for state, count in summary.items():
    if count > 0:
        print(f"  {state}: {count}")
if failures:
    print(f"\n  Failed: {failures}")

Status across 108 tasks:
  RUNNING: 2
  READY: 106


## 6. Verify outputs once complete

In [10]:
if IS_COLAB:
    FEATURE_STACK_DIR = Path("/content/drive/MyDrive/thesisv4/data/raw") / DRIVE_FOLDER
output_path = FEATURE_STACK_DIR
if output_path.exists():
    files = sorted(output_path.glob("features_m*.tif"))
    print(f"Output files found: {len(files)}")
    if len(files) > 0:
        print(f"First:  {files[0].name}")
        print(f"Last:   {files[-1].name}")
        total_size = sum(f.stat().st_size for f in files)
        print(f"Total size: {total_size / (1024 ** 3):.2f} GB")

        present_m_idx = set()
        for f in files:
            stem = f.stem
            try:
                m_idx = int(stem.split("_m")[1])
                present_m_idx.add(m_idx)
            except (IndexError, ValueError):
                continue

        expected = set(range(T)) - {
            m_idx for m_idx, ym in enumerate(all_months) if ym in missing}
        missing_files = expected - present_m_idx
        if missing_files:
            print(f"  Still missing m_idx: {sorted(missing_files)}")
        else:
            print(f"  All expected m_idx present (excluding the "
                  f"{len(missing)} unavailable months).")
else:
    print(f"Output folder not yet created. Tasks may still be running.")
    print(f"Expected location: {output_path}")

Output files found: 108
First:  features_m000.tif
Last:   features_m107.tif
Total size: 35.65 GB
  All expected m_idx present (excluding the 0 unavailable months).
